In [2]:
## implement naive bayes with tips data

import seaborn as sns
data = sns.load_dataset('tips')
print(data.head())
from sklearn.model_selection import train_test_split

   total_bill   tip     sex smoker  day    time  size
0       16.99  1.01  Female     No  Sun  Dinner     2
1       10.34  1.66    Male     No  Sun  Dinner     3
2       21.01  3.50    Male     No  Sun  Dinner     3
3       23.68  3.31    Male     No  Sun  Dinner     2
4       24.59  3.61  Female     No  Sun  Dinner     4


In [5]:
import warnings
warnings.filterwarnings('ignore')

In [6]:
data['sex'].value_counts()

sex
Male      157
Female     87
Name: count, dtype: int64

In [7]:
print(data['smoker'].value_counts())

smoker
No     151
Yes     93
Name: count, dtype: int64


In [9]:
print(data['day'].value_counts())

day
Sat     87
Sun     76
Thur    62
Fri     19
Name: count, dtype: int64


In [10]:
data['time'].value_counts()

time
Dinner    176
Lunch      68
Name: count, dtype: int64

In [12]:
data['tip'].value_counts()

tip
2.00    33
3.00    23
4.00    12
5.00    10
2.50    10
        ..
1.47     1
1.17     1
4.67     1
5.92     1
1.75     1
Name: count, Length: 123, dtype: int64

In [3]:
X = data[['sex','day','time','smoker','tip','size']]
y = data['total_bill']

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:
X_train

,sex,day,time,smoker,tip,size
228,Male,Sat,Dinner,No,2.72,2
208,Male,Sat,Dinner,Yes,2.03,2
96,Male,Fri,Dinner,Yes,4.00,2
167,Male,Sun,Dinner,No,4.50,4
84,Male,Thur,Lunch,No,2.03,2
...,...,...,...,...,...,...
106,Male,Sat,Dinner,Yes,4.06,2
14,Female,Sun,Dinner,No,3.02,2
92,Female,Fri,Dinner,Yes,1.00,2
179,Male,Sun,Dinner,Yes,3.55,2


In [16]:
y_train

228    13.28
208    24.27
96     27.28
167    31.71
84     15.98
       ...  
106    20.49
14     14.83
92      5.75
179    34.63
102    44.30
Name: total_bill, Length: 195, dtype: float64

In [6]:
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

le1 = LabelEncoder()
le2 = LabelEncoder()
le3 = LabelEncoder()

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

categorical_features = ['sex', 'day', 'time', 'smoker']
numeric_features = ['tip', 'size']

In [25]:
X_train

,sex,day,time,smoker,tip,size
228,Male,Sat,Dinner,No,2.72,2
208,Male,Sat,Dinner,Yes,2.03,2
96,Male,Fri,Dinner,Yes,4.00,2
167,Male,Sun,Dinner,No,4.50,4
84,Male,Thur,Lunch,No,2.03,2
...,...,...,...,...,...,...
106,Male,Sat,Dinner,Yes,4.06,2
14,Female,Sun,Dinner,No,3.02,2
92,Female,Fri,Dinner,Yes,1.00,2
179,Male,Sun,Dinner,Yes,3.55,2


In [7]:
ct = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_features),
    ],
    remainder='passthrough'
)

In [8]:
import sys
import numpy as np
np.set_printoptions(threshold=sys.maxsize)
X_train = np.asarray(ct.fit_transform(X_train), dtype=float)
X_test = np.asarray(ct.transform(X_test), dtype=float)

In [9]:
from sklearn.svm import SVR
svr = SVR(kernel='linear')
svr.fit(X_train,y_train)
y_pred = svr.predict(X_test)

In [34]:
y_pred

array([17.47730511, 13.22961273, 19.6704074 , 31.49927879, 14.21398425,
       14.69659439, 16.11792082, 14.3527514 , 17.02782624, 18.48418619,
       16.3175912 , 12.19686193, 11.20301336, 14.69659439,  9.00164892,
       13.99109371, 23.78500714, 19.15853138, 15.11017177, 31.84679862,
       21.48920687, 21.22265878, 21.30176048, 12.13510236, 22.37429079,
       13.87894454, 12.33409898, 24.38973894, 19.6704074 , 35.37513747,
       22.625967  , 13.57120423, 21.32419787, 19.03692612, 20.90730914,
       22.02873745, 14.02688712, 29.91871432, 15.06298644, 15.30047217,
       11.51750667, 12.74700259, 14.92800089, 15.25630855, 14.38865288,
        8.81742775, 13.92149195, 18.01116118, 11.19240328])

In [35]:
from sklearn.metrics import mean_squared_error, r2_score
print("mse",mean_squared_error(y_test, y_pred))
print("r2",r2_score(y_test, y_pred))

mse 28.174532852255382
r2 0.667708259358899


In [11]:
from sklearn.model_selection import GridSearchCV

params = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 0.01, 0.1],
    'kernel': ['rbf', 'linear']
}

grid = GridSearchCV(
    svr,
    param_grid=params,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)
grid.fit(X_train, y_train)
y_pred_grid = grid.predict(X_test)
y_pred_grid

Fitting 5 folds for each of 24 candidates, totalling 120 fits


array([17.34767775, 13.21903573, 19.55058534, 31.88976857, 14.61688419,
       15.1911252 , 16.54447094, 14.28317599, 17.09443757, 20.74607469,
       16.36976893, 11.95992397, 11.06942812, 15.1911252 ,  8.94339527,
       13.79410028, 24.24730156, 19.26844348, 14.94748102, 31.75749491,
       21.21601597, 21.0490772 , 21.74019229, 11.89690931, 22.91982365,
       13.78469817, 12.3053231 , 24.89092549, 19.55058534, 35.33433457,
       22.68627694, 13.83749144, 21.03261147, 19.41428921, 20.94773419,
       21.87501888, 16.18315104, 30.40063341, 15.00784463, 15.60190204,
       11.26676267, 12.64479472, 14.95193899, 15.18189243, 14.78006035,
        9.12221787, 14.20993994, 17.77231467, 11.04356028])

In [12]:
grid.best_params_

{'C': 100, 'gamma': 'scale', 'kernel': 'linear'}

In [13]:
grid.best_score_

np.float64(0.5053770439370198)

In [15]:
from sklearn.metrics import mean_squared_error, r2_score
print("mse",mean_squared_error(y_test, y_pred_grid))
print("r2",r2_score(y_test, y_pred_grid))

mse 28.78269773491713
r2 0.6605355346675547
